# 🚫 Hate Speech Detection — Exploratory Data Analysis

**Project:** Hate Speech Detection | Data Science Industrial Program — 1stop.ai / Personifwy
**Author:** Agam Saxena


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/raw/hate_speech.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Column types:\n', df.dtypes)
print('\nNull values:\n', df.isnull().sum())

## 2. Class Distribution

In [ ]:
df['binary_label'] = df['class'].apply(lambda x: 1 if x == 0 else 0)
label_counts = df['binary_label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
sns.countplot(x='binary_label', data=df, palette=['steelblue','crimson'], ax=axes[0])
axes[0].set_title('Class Distribution (Binary)')
axes[0].set_xticklabels(['Not Hate (0)', 'Hate Speech (1)'])

# Pie chart
axes[1].pie(label_counts, labels=['Not Hate', 'Hate Speech'], autopct='%1.1f%%',
            colors=['steelblue','crimson'], startangle=90)
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.show()
print(f'Class imbalance ratio: {label_counts[0]/label_counts[1]:.1f}:1')

## 3. Text Length Analysis

In [ ]:
df['text_col'] = df['tweet'] if 'tweet' in df.columns else df.iloc[:, 0]
df['text_length'] = df['text_col'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['text_col'].apply(lambda x: len(str(x)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, color in [(0, 'steelblue'), (1, 'crimson')]:
    subset = df[df['binary_label'] == label]['text_length']
    axes[0].hist(subset, bins=30, alpha=0.6, color=color,
                 label='Not Hate' if label == 0 else 'Hate Speech')
axes[0].set_title('Word Count Distribution by Class')
axes[0].set_xlabel('Word Count')
axes[0].legend()

sns.boxplot(x='binary_label', y='text_length', data=df,
            palette=['steelblue','crimson'], ax=axes[1])
axes[1].set_title('Word Count Boxplot by Class')
axes[1].set_xticklabels(['Not Hate', 'Hate Speech'])
plt.tight_layout()
plt.show()

df.groupby('binary_label')['text_length'].describe()

## 4. Most Frequent Words (per class)

In [ ]:
import sys, os
sys.path.insert(0, '../src')
from preprocess import clean_text

def top_words(texts, n=20):
    all_words = ' '.join(texts.apply(clean_text)).split()
    return Counter(all_words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, color, title in [
    (axes[0], 0, 'steelblue', 'Top Words — Not Hate Speech'),
    (axes[1], 1, 'crimson',   'Top Words — Hate Speech')
]:
    words, counts = zip(*top_words(df[df['binary_label']==label]['text_col']))
    ax.barh(list(words)[::-1], list(counts)[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.show()

## 5. Summary & Key Findings

In [ ]:
print('=== EDA Summary ===')
print(f'Total samples        : {len(df)}')
print(f'Hate speech samples  : {(df["binary_label"]==1).sum()}')
print(f'Non-hate samples     : {(df["binary_label"]==0).sum()}')
print(f'Avg words (hate)     : {df[df["binary_label"]==1]["text_length"].mean():.1f}')
print(f'Avg words (not hate) : {df[df["binary_label"]==0]["text_length"].mean():.1f}')
print(f'Null values          : {df.isnull().sum().sum()}')
print('\nKey Observations:')
print('  • Significant class imbalance → handled with SMOTE in training')
print('  • Hate speech tweets tend to be shorter on average')
print('  • Preprocessing removes noise (URLs, mentions, punctuation) effectively')